# Agglomerative Clustering in JAX

This notebook demonstrates hierarchical agglomerative clustering implemented in JAX.

The example performs bottom-up merging using single-link distance.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(12)
centers = jnp.array([[-4.0,-2.0],[0.0,4.0],[4.0,-1.0]])
samples = []
for i, c in enumerate(centers):
    key, sub = jax.random.split(key)
    samples.append(jax.random.normal(sub, (120,2)) * 0.8 + c)
X = jnp.concatenate(samples, axis=0)
X = X[jax.random.permutation(key, X.shape[0])]

In [ ]:
def pairwise_distances(X):
    dif = X[:, None, :] - X[None, :, :]
    return jnp.sqrt(jnp.sum(dif ** 2, axis=2))

def agglomerative_single_link(X, target_clusters=3):
    n = X.shape[0]
    clusters = {i: [i] for i in range(n)}
    distances = pairwise_distances(X).tolist()
    while len(clusters) > target_clusters:
        min_d = float('inf')
        pair = None
        keys = list(clusters.keys())
        for i in range(len(keys)):
            for j in range(i+1, len(keys)):
                ci = clusters[keys[i]]
                cj = clusters[keys[j]]
                d = min(distances[a][b] for a in ci for b in cj)
                if d < min_d:
                    min_d = d
                    pair = (keys[i], keys[j])
        a, b = pair
        clusters[a] = clusters[a] + clusters[b]
        del clusters[b]
    labels = jnp.zeros((n,), dtype=jnp.int32)
    for idx, k in enumerate(clusters.keys()):
        for v in clusters[k]:
            labels = labels.at[v].set(idx)
    return labels

labels = agglomerative_single_link(X, target_clusters=3)
labels

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(X[:,0], X[:,1], c=labels, cmap='tab10')
plt.title('Agglomerative Clustering (single-link)')
plt.show()